# 02 May 05 Linear Clip — LR & LR decay search

Фиксируем **LinearBidder bin clip** (как в старом «fixed baseline»), а через Optuna перебираем **скидку MDP для DQN (`dqn_gamma`)**, **стартовые lr** для DQN и RewardNet и **стратегии затухания learning rate**:

- **DQN `dqn_gamma`**: сетка **1.0, 0.999, 0.99** (дисконт по горизонту Q-learning).
- **DQN LR decay**: `ExponentialLR` (несколько множителей по LR), либо без scheduler, либо `CosineAnnealingWarmRestarts` с периодами 2000 / 8000 шагов обучения (один `scheduler.step` на один gradient step).
- **RewardNet**: без decay или `ExponentialLR` с $\gamma \in \{0.9995, 0.9999\}$.

Профиль: `may06_default_linear_clip_lr_scheduler_search` в `profiles.py` (`n_trials=36`). Число триалов можно переопределить константой ниже.

In [1]:
import sys
import pickle
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
linear_tuned_params

{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [3]:
RUN_NAME = 'may06_linear_clip_lr_scheduler_search'
DRLB_PROFILE = 'may05_default_linear_clip_lr_scheduler_search'
VERBOSE = False
SHOW_PROGRESS = True
# None = взять n_trials из профиля (36)
N_TRIALS_OVERRIDE = None

In [4]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
if N_TRIALS_OVERRIDE is not None:
    config = replace(config, n_trials=int(N_TRIALS_OVERRIDE))
config = replace(config, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['bid_lower_clip'] = 3
base_drlb_params['bid_upper_clip'] = 8
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    show_progress=SHOW_PROGRESS,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
tuning = summary['tuning']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'n_trials': tuning['n_trials'],
    'study_best_val_clicks': tuning['study_best_value'],
    'best_trial': tuning['best_trial_number'],
    'best_params': tuning['best_params'],
    'best_val_metrics': tuning['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}

autobidder_check campaigns: 100%|██████████| 257/257 [00:18<00:00, 13.53campaign/s, campaign_id=7.46e+7]
[I 2026-05-05 11:59:10,966] A new study created in memory with name: no-name-2484315a-02ef-4ac3-a4fd-99f5890f1fd2
Best trial: 0. Best value: 2001.57:   3%|▎         | 1/36 [03:37<2:07:00, 217.74s/it]

[I 2026-05-05 12:02:48,702] Trial 0 finished with value: 2001.5665756188923 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.003, 'reward_net_lr': 0.01, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'none'}. Best is trial 0 with value: 2001.5665756188923.


Best trial: 0. Best value: 2001.57:   6%|▌         | 2/36 [07:07<2:00:42, 213.03s/it]

[I 2026-05-05 12:06:18,437] Trial 1 finished with value: 1921.7589853826462 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.0001, 'reward_net_lr': 0.003, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'exp_0.9995'}. Best is trial 0 with value: 2001.5665756188923.


Best trial: 2. Best value: 2324.26:   8%|▊         | 3/36 [10:37<1:56:26, 211.70s/it]

...

[I 2026-05-05 13:42:36,530] Trial 34 finished with value: 2340.1159896762174 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.0003, 'reward_net_lr': 0.003, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'exp_0.9999'}. Best is trial 24 with value: 2466.06143886923.


Best trial: 24. Best value: 2466.06: 100%|██████████| 36/36 [1:46:01<00:00, 176.72s/it]


[I 2026-05-05 13:45:12,824] Trial 35 finished with value: 2021.3662406935068 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.003, 'reward_net_lr': 0.01, 'dqn_lr_decay': 'cosine_T2000', 'reward_net_lr_decay': 'exp_0.9995'}. Best is trial 24 with value: 2466.06143886923.


autobidder_check campaigns: 100%|██████████| 257/257 [00:16<00:00, 15.91campaign/s, campaign_id=7.46e+7]


{'run_name': 'may06_linear_clip_lr_scheduler_search',
 'profile': 'may06_default_linear_clip_lr_scheduler_search',
 'n_trials': 36,
 'study_best_val_clicks': 2466.06143886923,
 'best_trial': 24,
 'best_params': {'dqn_gamma': 1.0,
  'dqn_lr': 0.0003,
  'reward_net_lr': 0.01,
  'dqn_lr_decay': 'cosine_T2000',
  'reward_net_lr_decay': 'exp_0.9999'},
 'best_val_metrics': {'cpc_relative': 1051.5463522684306,
  'rmse': 1.2492047379067723,
  'clicks_sum': 2466.06143886923,
  'quickspend': 0.023346303501945526,
  'skipped_campaigns': 0,
  'time_inference_sec': 15.422693967819214,
  'time_overall_sec': 19.623852014541626,
  'average_end_balance_share': 0.7379224599189419,
  'label': 'best_val',
  'train_steps': 48240,
  'last_dqn_loss': 46.39968490600586,
  'last_reward_net_loss': 139.42835998535156,
  'dqn_loss_mean': 23.377484411404275,
  'dqn_loss_p95': 53.39493198394776,
  'reward_net_loss_mean': 240.92280504525235,
  'reward_net_loss_p95': 738.7282714843747,
  'reward_signal_mean': 9.02592

In [5]:
trials_df = pd.DataFrame(tuning['all_trials_summary'])
if 'clicks_sum' in trials_df.columns:
    trials_df = trials_df.sort_values('clicks_sum', ascending=False)
trials_df

,trial,dqn_gamma,dqn_lr,reward_net_lr,dqn_lr_decay,reward_net_lr_decay,rmse,clicks_sum,cpc_relative,quickspend,last_dqn_loss,last_reward_net_loss,dqn_loss_mean,reward_net_loss_mean
26,26,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
33,33,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
32,32,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
31,31,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
24,24,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
25,25,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
8,8,1.000,0.0003,0.0010,cosine_T2000,exp_0.9999,1.436881,2391.657496,818.878595,0.054475,25.303316,257.954254,21.704505,250.602151
34,34,1.000,0.0003,0.0030,cosine_T8000,exp_0.9999,1.255702,2340.115990,1207.533119,0.027237,24.510843,212.278915,26.648206,197.959975
18,18,1.000,0.0010,0.0100,cosine_T2000,exp_0.9999,1.401908,2326.004889,974.469456,0.058366,28.818624,117.121498,16.557584,255.321782
22,22,1.000,0.0010,0.0100,cosine_T2000,exp_0.9999,1.401908,2326.004889,974.469456,0.058366,28.818624,117.121498,16.557584,255.321782


In [6]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])

,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
